# Cash Flow Statement Analysis

*A comprehensive, from-scratch exploration of cash flow statement construction, classification, and analysis — aligned with CFA Level 1 curriculum.*

---

**What this notebook covers:**

| # | Topic |
|---|-------|
| 1 | Why cash flow matters |
| 2 | Setup |
| 3 | Operating activities — direct vs indirect method |
| 4 | Working capital adjustments in the indirect method |
| 5 | IFRS vs US GAAP classification differences |
| 6 | Free cash flow — FCFF and FCFE |
| 7 | Cash flow quality indicators |
| 8 | Company lifecycle stage analysis |
| 9 | Comprehensive worked example |
| 10 | References |

## 1. Why Cash Flow Matters

### "Cash Is Fact, Profit Is Opinion"

The income statement is prepared on an **accrual basis**: revenues are recognised when earned and expenses when incurred, regardless of when cash actually changes hands. This creates an inherent gap between **reported earnings** and **actual cash generation**. A company can report rising profits while simultaneously running out of cash — a phenomenon that has preceded some of the most spectacular corporate failures in history (Enron, WorldCom, Wirecard).

The **cash flow statement** closes this gap by reporting what actually happened to a company's cash during the period. It answers three fundamental questions:

1. **How much cash did the business generate from its core operations?**
2. **How much cash was invested in (or realised from) long-term assets?**
3. **How was the business financed — through debt, equity, or returning capital to shareholders?**

Unlike the income statement, the cash flow statement cannot be easily manipulated through accounting policy choices. Depreciation methods, revenue recognition timing, and inventory costing assumptions all affect net income, but they do not affect the amount of cash that actually moved through the company's bank accounts.

> **Key Concept:** The cash flow statement is the only financial statement that is essentially immune to accounting manipulation. You can argue about revenue recognition policies or depreciation methods, but cash either moved or it didn't. This makes it the single most important statement for assessing solvency and earnings quality.

### The Three Activities

Every cash flow is classified into one of three buckets:

| Activity | Abbreviation | What it captures | Examples |
|----------|-------------|-----------------|----------|
| **Operating** | CFO | Cash from day-to-day business | Cash from customers, payments to suppliers, wages, taxes |
| **Investing** | CFI | Cash for long-term asset base | Purchase/sale of PP&E, acquisitions, investment securities |
| **Financing** | CFF | Cash from/to capital providers | Debt issuance/repayment, equity issuance/buybacks, dividends |

The three activities are connected by a simple identity:

$$\Delta \text{Cash} = \text{CFO} + \text{CFI} + \text{CFF}$$

This identity must hold exactly. The change in cash on the balance sheet must equal the sum of the three activities on the cash flow statement. If it doesn't, there is an error. This serves as a powerful internal consistency check that auditors and analysts rely upon.

> **CFA Exam Tip:** The CFA exam frequently tests your ability to classify a specific cash flow into the correct activity. A useful heuristic: if the item appears on the income statement, it's likely operating; if it relates to a long-term asset, it's investing; if it relates to long-term liabilities or equity, it's financing. There are important exceptions (interest, dividends) which differ between IFRS and US GAAP — we cover these in Section 5.

### Why Analysts Focus on Cash Flow

There are several reasons why analysts often trust the cash flow statement more than the income statement:

1. **Solvency assessment**: A company that cannot generate positive operating cash flow will eventually become insolvent, regardless of its reported profits. Cash is what pays suppliers, employees, and lenders — not "earnings."

2. **Earnings quality**: Persistent divergence between net income and CFO is a red flag that earnings may be of low quality — propped up by aggressive accounting rather than genuine economic activity. Academic research (Sloan, 1996) has shown that stocks with high accruals (i.e., large NI-CFO gaps) tend to underperform.

3. **Valuation**: Discounted cash flow (DCF) models, the gold standard of intrinsic valuation, require cash flow inputs — not accounting earnings. Free cash flow to the firm (FCFF) or free cash flow to equity (FCFE) are derived from the cash flow statement.

4. **Dividend sustainability**: Dividends are paid in cash, not in "earnings". A company can only sustain its dividend if CFO (plus any financing) covers the payout. Many dividend cuts have been preceded by deteriorating CFO even as net income remained stable.

5. **Comparability**: Different companies may use different accounting policies for depreciation, revenue recognition, and inventory valuation. Cash flow strips away these choices and shows what actually happened, making cross-company comparison more meaningful.

6. **Fraud detection**: Many accounting frauds have been detectable through careful analysis of the cash flow statement. Inflated revenue that is never collected shows up as ballooning receivables and weak CFO. Capitalised operating expenses shift cash outflows from CFO to CFI, but total cash flow still tells the truth.

> **Common Mistake:** Students sometimes assume that a company with positive net income is financially healthy. This is not necessarily true. A company with $100M in net income but -$50M in CFO is in a precarious position — it is "earning" money on paper but consuming cash in practice. The reverse can also be true: a company with negative net income but positive CFO may be in better shape than it appears (common during heavy depreciation periods).

## 2. Setup

We import the standard scientific computing stack and configure consistent styling for all visualisations throughout this notebook.

In [ ]:
%matplotlib inline
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings

warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
rng = np.random.default_rng(SEED)

# Tolerance for numerical checks
ATOL = 1e-8
RTOL = 1e-6

# Colour palette
PRIMARY   = 'steelblue'
SECONDARY = 'coral'
TERTIARY  = 'seagreen'
ACCENT    = 'gold'

# Matplotlib defaults
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print("Setup complete.")

## 3. Operating Activities — Direct vs Indirect Method

### Overview

There are two methods for presenting cash flow from operations (CFO):

**Direct method**: Reports each major class of gross cash receipts and gross cash payments. It shows the actual cash collected from customers, cash paid to suppliers, cash paid for wages, etc. This method provides the most intuitive view of operating cash flows because it mirrors how a business owner thinks about cash: "How much came in from customers? How much went out to suppliers?"

**Indirect method**: Starts with net income and adjusts for non-cash items and working capital changes to arrive at CFO. It is a *reconciliation* from accrual-basis net income to cash-basis operating cash flow. This method is less intuitive but far more common, and it directly highlights the sources of divergence between earnings and cash flow.

> **Key Concept:** Both methods produce the **exact same CFO figure**. They differ only in *presentation*, not in substance. The direct method shows you *where the cash came from*; the indirect method shows you *why CFO differs from net income*.

### The Direct Method in Detail

Under the direct method, each line item from the income statement is converted to its cash equivalent using balance sheet changes. The general formula is:

$$\text{CFO}_{\text{direct}} = \text{Cash from customers} - \text{Cash to suppliers} - \text{Cash for operating expenses} - \text{Cash for taxes} - \text{Cash for interest}$$

The conversion from accrual to cash for each line item follows a specific pattern:

| Income Statement Item | Balance Sheet Adjustment | Cash Flow Item |
|----------------------|-------------------------|----------------|
| Revenue | $-\Delta$ Accounts Receivable | Cash from customers |
| COGS | $+\Delta$ Accounts Payable $-\Delta$ Inventory | Cash to suppliers |
| Operating Expenses | $+\Delta$ Accrued Expenses $-\Delta$ Prepaid Expenses | Cash for operating expenses |
| Tax Expense | $+\Delta$ Tax Payable | Cash for taxes |
| Interest Expense | $+\Delta$ Interest Payable | Cash for interest |

**Example — Cash collected from customers:**
If a company reports $10,000 in revenue and accounts receivable increased by $500 during the period, then cash actually collected from customers is $10,000 - $500 = $9,500. The remaining $500 was "earned" on paper but not yet received as cash.

**Example — Cash paid to suppliers:**
If COGS is $6,000, inventory increased by $300, and accounts payable increased by $200, then cash paid to suppliers = $6,000 + $300 - $200 = $6,100. The company bought $300 more than it sold (inventory build), but $200 of purchases were on credit (AP increase).

### The Indirect Method in Detail

Under the indirect method, we start with net income and systematically undo all the accrual accounting that separates NI from cash. The formula is:

$$\text{CFO}_{\text{indirect}} = \text{Net Income} + \text{Non-cash charges} \pm \text{Working capital changes} - \text{Non-operating gains} + \text{Non-operating losses}$$

The major adjustment categories are:

1. **Add back non-cash expenses**: Depreciation and amortisation are the most significant. These charges reduce net income but involve zero cash outflow (the cash was spent when the asset was originally purchased). Other non-cash items include stock-based compensation, impairment charges, and deferred tax adjustments.

2. **Remove non-operating items**: Gains and losses on the sale of assets must be removed from CFO because the total cash proceeds appear in the investing section (CFI). If we left the gain in CFO *and* put the full proceeds in CFI, we would double-count. For a gain, we subtract it from NI (removing the income statement boost); for a loss, we add it back.

3. **Adjust for working capital changes**: Changes in current operating assets and current operating liabilities reflect the difference between accrual and cash recognition. These are covered extensively in Section 4.

### Why the Indirect Method Dominates

In practice, approximately 99% of public companies use the indirect method. The reasons are entirely practical:

- The indirect method is **easier to prepare** because it starts from readily available income statement data and only requires balance sheet comparisons
- Under US GAAP (ASC 230), if a company uses the direct method, it must *also* provide the indirect method reconciliation as a supplemental schedule — so the direct method creates extra work with no reduction in required disclosures
- The indirect method **directly highlights** the sources of divergence between net income and cash flow, which is exactly what analysts want to understand
- Most financial databases (Bloomberg, FactSet, S&P Capital IQ) store and display the indirect method by default

> **CFA Exam Tip:** Even though the indirect method is far more common in practice, the CFA exam expects you to be fluent in both methods. You may be asked to convert between them, or to compute cash collected from customers or cash paid to suppliers from income statement and balance sheet data. Master the conversion formulas in the direct method table above.

### Mathematical Proof of Equivalence

We can prove rigorously that both methods yield identical CFO. Consider a simplified income statement with Revenue (R), COGS (C), Operating Expenses (O), and no non-cash items or non-operating items for clarity:

$$\text{NI} = R - C - O$$

**Direct method:**

$$\text{CFO}_{\text{direct}} = \underbrace{(R - \Delta AR)}_{\text{Cash from customers}} - \underbrace{(C + \Delta \text{Inv} - \Delta AP)}_{\text{Cash to suppliers}} - \underbrace{(O - \Delta \text{Accruals})}_{\text{Cash for OpEx}}$$

Expanding:

$$= R - \Delta AR - C - \Delta \text{Inv} + \Delta AP - O + \Delta \text{Accruals}$$

Regrouping:

$$= \underbrace{(R - C - O)}_{\text{Net Income}} + \underbrace{(-\Delta AR - \Delta \text{Inv} + \Delta AP + \Delta \text{Accruals})}_{\text{Working capital adjustments}}$$

$$= \text{NI} + \text{WC adjustments} = \text{CFO}_{\text{indirect}}$$

$\blacksquare$

This proof extends naturally to include depreciation (a non-cash charge added back in indirect, absent from direct since it never involves cash) and gains/losses on asset sales (removed in indirect, absent from direct since only actual cash proceeds matter).

### Sample Data for Demonstration

We now create a sample income statement and balance sheet to demonstrate both methods and verify their equivalence numerically.

In [ ]:
# ── Sample income statement and balance sheet data ──────────────────────

# Income statement (Year 1)
revenue = 5000.0
cogs = -3000.0
depreciation = -400.0
operating_expenses = -600.0  # SG&A excluding depreciation
interest_expense = -100.0
tax_expense = -225.0
gain_on_sale = 50.0  # gain on sale of equipment (non-operating)

net_income = revenue + cogs + depreciation + operating_expenses + interest_expense + tax_expense + gain_on_sale

print("=" * 55)
print("INCOME STATEMENT")
print("=" * 55)
print(f"  Revenue                     {revenue:>10,.0f}")
print(f"  Cost of Goods Sold          {cogs:>10,.0f}")
print(f"  Gross Profit                {revenue + cogs:>10,.0f}")
print(f"  Depreciation                {depreciation:>10,.0f}")
print(f"  Operating Expenses (SG&A)   {operating_expenses:>10,.0f}")
print(f"  Interest Expense            {interest_expense:>10,.0f}")
print(f"  Gain on Sale of Equipment   {gain_on_sale:>10,.0f}")
print(f"  Tax Expense                 {tax_expense:>10,.0f}")
print(f"  {'─' * 35}")
print(f"  Net Income                  {net_income:>10,.0f}")

# Balance sheet changes (Year 0 → Year 1)
delta_ar = 150.0        # accounts receivable increased
delta_inventory = 80.0  # inventory increased
delta_ap = 60.0         # accounts payable increased
delta_accrued = 30.0    # accrued expenses increased
delta_tax_payable = -20.0  # taxes payable decreased
delta_interest_payable = 10.0  # interest payable increased

print(f"\nBalance sheet changes (Year 0 -> Year 1):")
print(f"  Accounts Receivable    {delta_ar:>+10,.0f}")
print(f"  Inventory              {delta_inventory:>+10,.0f}")
print(f"  Accounts Payable       {delta_ap:>+10,.0f}")
print(f"  Accrued Expenses       {delta_accrued:>+10,.0f}")
print(f"  Tax Payable            {delta_tax_payable:>+10,.0f}")
print(f"  Interest Payable       {delta_interest_payable:>+10,.0f}")

### Indirect Method Computation

We now apply the indirect method step by step. Starting from net income, we:
1. Add back depreciation (non-cash)
2. Subtract the gain on sale (non-operating — belongs in CFI)
3. Apply working capital adjustments

In [ ]:
# ── Indirect method ──────────────────────────────────────────────────────

print("=" * 55)
print("CFO -- INDIRECT METHOD")
print("=" * 55)
print(f"  Net Income                  {net_income:>10,.0f}")
print(f"  Adjustments for non-cash items:")
print(f"    + Depreciation            {-depreciation:>10,.0f}")
print(f"    - Gain on sale            {-gain_on_sale:>10,.0f}")

# Working capital adjustments
wc_ar = -delta_ar
wc_inv = -delta_inventory
wc_ap = delta_ap
wc_accrued = delta_accrued
wc_tax = delta_tax_payable
wc_interest = delta_interest_payable

print(f"  Working capital changes:")
print(f"    Accounts Receivable       {wc_ar:>10,.0f}")
print(f"    Inventory                 {wc_inv:>10,.0f}")
print(f"    Accounts Payable          {wc_ap:>10,.0f}")
print(f"    Accrued Expenses          {wc_accrued:>10,.0f}")
print(f"    Tax Payable               {wc_tax:>10,.0f}")
print(f"    Interest Payable          {wc_interest:>10,.0f}")

cfo_indirect = (net_income
                + (-depreciation)
                + (-gain_on_sale)
                + wc_ar + wc_inv + wc_ap + wc_accrued + wc_tax + wc_interest)

print(f"  {'─' * 35}")
print(f"  CFO (Indirect)              {cfo_indirect:>10,.0f}")

### Direct Method Computation

Now we compute the same CFO using the direct method, converting each income statement line to its cash equivalent.

In [ ]:
# ── Direct method ────────────────────────────────────────────────────────

cash_from_customers = revenue - delta_ar
cash_to_suppliers = -((-cogs) + delta_inventory - delta_ap)
cash_for_opex = -((-operating_expenses) - delta_accrued)
cash_for_interest = -((-interest_expense) - delta_interest_payable)
cash_for_taxes = -((-tax_expense) - delta_tax_payable)

cfo_direct = (cash_from_customers + cash_to_suppliers
              + cash_for_opex + cash_for_interest + cash_for_taxes)

print("=" * 55)
print("CFO -- DIRECT METHOD")
print("=" * 55)
print(f"  Cash from customers         {cash_from_customers:>10,.0f}")
print(f"  Cash to suppliers           {cash_to_suppliers:>10,.0f}")
print(f"  Cash for operating expenses {cash_for_opex:>10,.0f}")
print(f"  Cash for interest           {cash_for_interest:>10,.0f}")
print(f"  Cash for taxes              {cash_for_taxes:>10,.0f}")
print(f"  {'─' * 35}")
print(f"  CFO (Direct)                {cfo_direct:>10,.0f}")

# Verify equivalence
print(f"\n  Indirect = Direct? {np.isclose(cfo_indirect, cfo_direct, atol=ATOL)}")

## 4. Working Capital Adjustments in the Indirect Method

### The Core Principle

The most confusing aspect of the indirect method for students is understanding why some balance sheet changes are added and others subtracted. The key insight is this:

**The indirect method starts with net income (accrual basis) and converts it to cash basis.**

To convert from accrual to cash, we must undo the effects of timing differences. When revenue is recognised but cash hasn't arrived, we must subtract. When an expense is recognised but cash hasn't left, we must add back.

### The Sign Rules

When a current asset *increases*, it means the company spent (or failed to collect) cash that was already counted as revenue or not yet counted as expense. Therefore, an increase in a current asset is a **cash use** (subtracted from NI).

When a current liability *increases*, it means the company received cash (or avoided paying cash) that was counted as an expense. Therefore, an increase in a current liability is a **cash source** (added to NI).

> **Key Concept:** The rule is simple — for **current assets**, reverse the sign (increase = subtract, decrease = add). For **current liabilities**, keep the sign (increase = add, decrease = subtract). The mnemonic: *assets and cash move in opposite directions; liabilities and cash move in the same direction.*

### Comprehensive Table of Every Adjustment

| Balance Sheet Item | Change | Effect on Cash | Intuitive Explanation |
|-------------------|--------|---------------|----------------------|
| **Accounts Receivable** | Increase | **Outflow** (subtract) | Revenue was recognised but cash was NOT collected. Revenue inflated NI relative to actual cash received. |
| **Accounts Receivable** | Decrease | **Inflow** (add) | Cash was collected from prior-period sales not in current NI. More cash came in than revenue suggests. |
| **Inventory** | Increase | **Outflow** (subtract) | Company purchased more inventory than it sold (COGS). Cash left the company but was not yet expensed through COGS. |
| **Inventory** | Decrease | **Inflow** (add) | Company sold from existing stock without purchasing replacements. COGS was charged but no new cash was spent on inventory. |
| **Prepaid Expenses** | Increase | **Outflow** (subtract) | Cash was paid in advance for expenses not yet recognised on the income statement. NI does not yet reflect this cash outflow. |
| **Prepaid Expenses** | Decrease | **Inflow** (add) | A prior cash payment is now being expensed. The expense reduces NI, but no cash left the company this period. |
| **Accounts Payable** | Increase | **Inflow** (add) | Expenses (COGS) were recognised (reducing NI) but NOT yet paid in cash. Company retains the cash. |
| **Accounts Payable** | Decrease | **Outflow** (subtract) | Company paid off prior-period obligations. Cash left, but no expense was charged to current NI. |
| **Accrued Liabilities** | Increase | **Inflow** (add) | Expenses accrued (wages, utilities) but not yet paid. NI is reduced by the expense but cash is retained. |
| **Accrued Liabilities** | Decrease | **Outflow** (subtract) | Prior-period accruals were paid in cash. Cash left but no current expense was recognised. |
| **Deferred Revenue** | Increase | **Inflow** (add) | Cash received from customers for services not yet delivered. Cash came in, but no revenue was recognised in NI. |
| **Deferred Revenue** | Decrease | **Outflow** (subtract) | Prior-period cash is now being recognised as revenue, inflating NI without new cash inflow. |
| **Taxes Payable** | Increase | **Inflow** (add) | Tax expense recognised but not yet paid. NI is reduced but cash is retained. |
| **Taxes Payable** | Decrease | **Outflow** (subtract) | Prior-period tax obligations paid. Cash outflow with no current-period expense. |

> **CFA Exam Tip:** A quick shortcut for the exam — for any current operating asset, *subtract the increase* (or *add the decrease*). For any current operating liability, *add the increase* (or *subtract the decrease*). Non-operating items (like short-term debt or current portion of long-term debt) go into CFF, not CFO.

### Worked Examples: Building Intuition

#### The Receivables Example

Consider a company that reports $1,000 in revenue. At the start of the year, accounts receivable was $200. At the end of the year, it is $350.

- Revenue (accrual) = $1,000 — this is what appears in net income
- $\Delta$AR = $350 - $200 = +$150
- Cash actually collected = Revenue - $\Delta$AR = $1,000 - $150 = $850

So NI includes $1,000 of revenue, but only $850 of cash came in. The indirect method corrects for this by subtracting the $150 increase in AR. The $150 represents sales that were booked on credit — real economic activity occurred, but cash hasn't arrived yet.

#### The Payables Example

Consider a company that reports $600 in COGS. At the start of the year, accounts payable was $100. At the end of the year, it is $160.

- COGS (accrual) = $600 — this reduced net income by $600
- $\Delta$AP = $160 - $100 = +$60
- Cash actually paid to suppliers = COGS - $\Delta$AP = $600 - $60 = $540

NI was reduced by $600, but only $540 of cash actually left. The indirect method corrects by adding back the $60 increase in AP. The company effectively "borrowed" from its suppliers by delaying payment.

#### The Inventory Example

Consider a retailer with COGS of $800. Inventory at start = $500, at end = $700.

- COGS charged to income = $800
- But inventory grew by $200, meaning the company purchased $800 (COGS) + $200 (build) = $1,000 of goods
- Only $800 hit the income statement as expense
- Cash paid (ignoring payables) = $1,000 but NI only reflects $800 of cost
- Adjustment: subtract the $200 inventory increase to capture the full cash outflow

#### The Deferred Revenue Example

A software company sells annual subscriptions. It receives $1,200 in cash for a 12-month subscription starting July 1. By year-end:

- Cash received = $1,200
- Revenue recognised (6 months) = $600
- Deferred revenue increased by $600

NI only includes $600, but $1,200 of cash came in. Adding the $600 increase in deferred revenue bridges the gap.

> **Common Mistake:** Students often confuse the direction of working capital adjustments. Remember: we are trying to convert from accrual to cash. If accrual-basis overstates cash inflows (like uncollected revenue), we must subtract. If accrual-basis overstates cash outflows (like unpaid expenses), we must add back. Think of it as "correcting" NI towards reality.

In [ ]:
# ── Demonstrate working capital adjustments with detailed breakdown ──────

# Balance sheet data for two years
bs_items = ['Accounts Receivable', 'Inventory', 'Prepaid Expenses',
            'Accounts Payable', 'Accrued Liabilities', 'Deferred Revenue']
year0 = np.array([800, 1200, 100, 500, 200, 150])
year1 = np.array([950, 1120, 130, 560, 230, 120])

deltas = year1 - year0
is_asset = np.array([True, True, True, False, False, False])
cash_effects = np.where(is_asset, -deltas, deltas)

print("=" * 70)
print("WORKING CAPITAL ADJUSTMENTS -- DETAILED BREAKDOWN")
print("=" * 70)
print(f"{'Item':<25} {'Year 0':>8} {'Year 1':>8} {'Delta':>8} {'Cash Effect':>12}")
print("-" * 70)
for i, item in enumerate(bs_items):
    print(f"{item:<25} {year0[i]:>8,.0f} {year1[i]:>8,.0f} {deltas[i]:>+8,.0f} {cash_effects[i]:>+12,.0f}")
print("-" * 70)
total_wc = cash_effects.sum()
print(f"{'Total WC Adjustment':<25} {'':>8} {'':>8} {'':>8} {total_wc:>+12,.0f}")
print(f"\nInterpretation: Working capital changes "
      f"{'consumed' if total_wc < 0 else 'released'} "
      f"${abs(total_wc):,.0f} of cash.")

### Visualising Working Capital Adjustments

The chart below shows each working capital adjustment as a horizontal bar. Green bars represent cash inflows (favourable adjustments that increase CFO relative to NI), while coral bars represent cash outflows (unfavourable adjustments that decrease CFO relative to NI).

In [ ]:
# ── Visualise working capital adjustments as a horizontal bar chart ──────

fig, ax = plt.subplots(figsize=(10, 5))
colors = [SECONDARY if ce < 0 else TERTIARY for ce in cash_effects]
bars = ax.barh(bs_items, cash_effects, color=colors, edgecolor='white', linewidth=1.5)

ax.axvline(0, color='black', linewidth=0.8)
for bar, val in zip(bars, cash_effects):
    offset = 5 if val >= 0 else -5
    ha = 'left' if val >= 0 else 'right'
    ax.text(bar.get_width() + offset, bar.get_y() + bar.get_height()/2,
            f'{val:+,.0f}', va='center', ha=ha, fontweight='bold', fontsize=11)

ax.set_xlabel('Cash Effect ($)')
ax.set_title('Working Capital Adjustments to CFO')
inflow_patch = mpatches.Patch(color=TERTIARY, label='Cash Inflow')
outflow_patch = mpatches.Patch(color=SECONDARY, label='Cash Outflow')
ax.legend(handles=[inflow_patch, outflow_patch], loc='lower right')
plt.tight_layout()
plt.show()

## 5. IFRS vs US GAAP Classification Differences

### Why Classification Matters

Both IFRS (IAS 7) and US GAAP (ASC 230) require the same three-activity structure (Operating, Investing, Financing), but they differ significantly in **where certain cash flows are classified**. This creates a comparability problem: two companies with identical economic reality can report very different CFO, CFI, and CFF numbers simply because they follow different accounting standards.

For analysts comparing companies across borders — a European IFRS reporter versus an American US GAAP reporter — these classification differences must be understood and adjusted before any meaningful comparison of operating cash flow can be made.

### Detailed Classification Comparison

| Cash Flow Item | US GAAP (ASC 230) | IFRS (IAS 7) | Analytical Impact |
|---------------|-------------------|--------------|-------------------|
| **Interest paid** | Operating (mandatory) | Operating **or** Financing (company's choice) | If IFRS company puts in CFF, CFO is higher |
| **Interest received** | Operating (mandatory) | Operating **or** Investing (company's choice) | If IFRS company puts in CFI, CFO is lower |
| **Dividends paid** | Financing (mandatory) | Operating **or** Financing (company's choice) | If IFRS company puts in CFO, CFO is lower |
| **Dividends received** | Operating (mandatory) | Operating **or** Investing (company's choice) | If IFRS company puts in CFI, CFO is lower |
| **Income taxes** | Operating (mandatory) | Operating (default); allocate to CFI/CFF if directly identifiable with those activities | Rarely differs in practice |
| **Bank overdrafts** | Financing activity | May be included in Cash & Cash Equivalents if integral to cash management | Affects reported cash balance |

> **Key Concept:** Under US GAAP, there is no flexibility — interest and dividends received go to CFO, interest paid goes to CFO, and dividends paid go to CFF. Under IFRS, companies have a **choice** for each of these items, which means two IFRS-reporting companies may classify the same item differently. Once chosen, the classification must be applied consistently.

### Analytical Implications in Detail

The flexibility under IFRS has real consequences for financial analysis:

1. **CFO inflation**: An IFRS company that classifies interest paid as CFF (rather than CFO) will report a higher CFO than a comparable US GAAP company. For a company with $200M in annual interest expense, this is a $200M difference in reported CFO — enormously material.

2. **CFF deflation**: Moving interest paid to CFF makes financing activities look worse (more cash outflow), potentially obscuring the true net financing position.

3. **CFO/NI ratio distortion**: Quality metrics like CFO/NI are affected. An IFRS company that excludes interest from CFO will have a systematically higher CFO/NI ratio — not because of better quality, but because of classification choice.

4. **FCFF computation**: When computing FCFF from CFO, you must know whether interest is included in CFO. If it is (US GAAP), add back Int(1-t). If it isn't (IFRS choice to put in CFF), do NOT add it back — it was never deducted from CFO in the first place.

> **CFA Exam Tip:** The CFA exam loves to test IFRS vs GAAP classification. Memorise the table above. A common question format: "Under IFRS, a company classifies interest paid as a financing activity. Compared to US GAAP treatment, the company's CFO is *higher* and CFF is *lower* (more negative)." Another favourite: asking you to adjust an IFRS company's CFO to be comparable with US GAAP.

### Numerical Impact Example

Consider two identical companies — one reporting under US GAAP, one under IFRS. The IFRS company chooses to classify interest paid in CFF and dividends received in CFI:

| Item | US GAAP Treatment | IFRS Treatment |
|------|------------------|----------------|
| Operating cash before interest/dividends | 500 (in CFO) | 500 (in CFO) |
| Interest paid: -80 | In CFO | In CFF |
| Dividends received: +30 | In CFO | In CFI |
| **Reported CFO** | **450** | **500** |
| **Reported CFI** | (reported separately) | -30 less favourable |
| **Reported CFF** | (reported separately) | -80 more negative |

The IFRS company appears to generate 11% more operating cash flow ($500 vs $450) — but the economic reality is identical. This is purely a presentation difference.

To make these companies comparable, the analyst should either:
- Reclassify the IFRS company's interest paid back to CFO: $500 - $80 = $420... wait, that undershoots. More carefully: the IFRS company's CFO of $500 excludes interest paid and excludes dividends received. To convert to US GAAP basis: $500 - $80 + $30 = $450, matching exactly.

> **Common Mistake:** Analysts who fail to adjust for classification differences may mistakenly conclude that the IFRS company is operationally superior. Always normalise classification before comparing CFO across reporting standards. Check the notes to the financial statements to identify where interest and dividends are classified.

In [ ]:
# ── Visualise IFRS vs US GAAP classification impact ─────────────────────

categories = ['Base Operating\nCash Flow', 'Interest\nPaid', 'Dividends\nReceived', 'Reported\nCFO']

usgaap_values = [500, -80, 30, 450]
ifrs_values   = [500,   0,  0, 500]

x = np.arange(len(categories))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, usgaap_values, width, label='US GAAP CFO',
               color=PRIMARY, edgecolor='white', linewidth=1.5)
bars2 = ax.bar(x + width/2, ifrs_values, width, label='IFRS CFO',
               color=SECONDARY, edgecolor='white', linewidth=1.5)

for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        y_pos = height + 5 if height >= 0 else height - 15
        ax.text(bar.get_x() + bar.get_width()/2., y_pos,
                f'${height:,.0f}', ha='center', va='bottom', fontweight='bold')

ax.set_ylabel('Amount ($)')
ax.set_title('IFRS vs US GAAP: Same Company, Different CFO\n'
             '(IFRS puts interest paid in CFF, dividends received in CFI)')
ax.set_xticks(x)
ax.set_xticklabels(categories)
ax.legend()
ax.axhline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

print("The IFRS company reports CFO of $500 vs US GAAP $450.")
print("Difference = interest paid ($80 moved to CFF) - dividends received ($30 moved to CFI) = $50.")

## 6. Free Cash Flow — FCFF and FCFE

### What Is Free Cash Flow?

Free cash flow represents the cash available to distribute to a company's capital providers **after** all operating expenses and necessary reinvestment have been made. It strips away the accounting noise and answers the fundamental question: "After running the business and maintaining/expanding the asset base, how much cash is left over?"

There are two variants, each suited to a different valuation framework:

- **FCFF (Free Cash Flow to the Firm)**: Cash available to ALL capital providers — both debt holders and equity holders. This is the cash flow that would exist if the company had no debt. It is used to value the entire enterprise.

- **FCFE (Free Cash Flow to Equity)**: Cash available to EQUITY holders only, after all obligations to debt holders have been met. This is used to value the equity portion of the company.

> **Key Concept:** FCFF is the appropriate cash flow for enterprise value (EV) models where you discount at WACC. FCFE is appropriate for equity-value models where you discount at the cost of equity ($k_e$). Using FCFF with $k_e$ or FCFE with WACC is a common and serious valuation error.

### Deriving FCFF from Three Starting Points

One of the most important skills tested on the CFA exam is the ability to compute FCFF from different starting points and arrive at the same answer. Here are the three canonical derivations:

#### Starting Point 1: From CFO (Cash Flow from Operations)

$$\text{FCFF} = \text{CFO} + \text{Int}(1 - t) - \text{FCInv}$$

**Why add back after-tax interest?** Under US GAAP, interest expense is included in CFO (it reduces operating cash flow). But FCFF should be available to ALL providers — including the debt holders who receive interest. So we add it back. We use $(1-t)$ because interest is tax-deductible: the government effectively "pays" a portion of the interest through the tax shield.

**Why subtract FCInv?** Fixed capital investment (capex minus proceeds from asset sales) represents the cash spent to maintain and grow the asset base. This cash is not available for distribution.

#### Starting Point 2: From EBIT

$$\text{FCFF} = \text{EBIT}(1 - t) + \text{Dep} - \text{FCInv} - \text{WCInv}$$

**Intuition:** Start with after-tax operating profit (which excludes interest, making it already "available to all providers"). Add back depreciation (non-cash). Subtract capex and working capital investment (actual cash spent on the business).

#### Starting Point 3: From Net Income

$$\text{FCFF} = \text{NI} + \text{NCC} + \text{Int}(1 - t) - \text{FCInv} - \text{WCInv}$$

**Intuition:** Start with net income. Add back non-cash charges (NCC) to approximate CFO. Add back after-tax interest (NI already deducted interest, but FCFF is pre-debt). Subtract investments.

Note that NI + NCC + WCInv $\approx$ CFO (the indirect method!), so Starting Point 3 collapses to Starting Point 1, as expected.

> **CFA Exam Tip:** You MUST be able to compute FCFF from all three starting points and get the same answer. The exam will give you different data sets and expect you to pick the correct formula. If given CFO directly, use Starting Point 1 (simplest). If given EBIT, use Starting Point 2. If given NI with non-cash charges broken out, use Starting Point 3.

### Deriving FCFE from FCFF

The relationship between FCFF and FCFE accounts for cash flows between the firm and its debt holders:

$$\text{FCFE} = \text{FCFF} - \text{Int}(1 - t) + \text{Net Borrowing}$$

Or equivalently, starting directly from CFO (under US GAAP, where interest is already in CFO):

$$\text{FCFE} = \text{CFO} - \text{FCInv} + \text{Net Borrowing}$$

This second formula is intuitive: start with operating cash flow, subtract what must be reinvested, add any net new debt raised. What remains is available to equity holders.

**Net Borrowing** = New debt issued - Debt repaid. If positive, the company raised more debt than it repaid, providing additional cash to equity holders. If negative, debt repayment consumed cash that might otherwise have gone to equity holders.

### Important Subtleties

1. **Maintenance vs Growth Capex**: In theory, only maintenance capex should be subtracted for "sustainable" free cash flow. In practice, companies rarely disclose the split, so analysts use total capex.

2. **Working Capital Investment**: For FCFF from EBIT or NI, you must separately account for WCInv. For FCFF from CFO, the working capital changes are already embedded in CFO.

3. **Preferred Dividends**: If the company has preferred stock, FCFE to common equity should subtract preferred dividends.

4. **Non-recurring Items**: One-time restructuring charges, litigation settlements, or asset write-downs should be evaluated for whether they represent ongoing cash needs.

> **Common Mistake:** When computing FCFF from CFO, students often forget to add back the after-tax interest expense. Remember: CFO has interest deducted (under US GAAP), but FCFF must be available to ALL providers including debt holders. If working under IFRS where the company classifies interest in CFF, do NOT add it back — it was never in CFO to begin with.

In [ ]:
# ── Compute FCFF and FCFE from sample data ──────────────────────────────

# Given data
ni = 725.0
dep_amort = 400.0
interest_exp = 100.0
tax_rate = 0.25
capex = -600.0       # negative = cash outflow
proceeds_sale = 50.0
wc_investment = -90.0  # negative = cash consumed by WC
net_borrowing = 200.0

# Back-solve EBIT: NI = (EBIT - Int)(1-t) => EBIT = NI/(1-t) + Int
ebit = ni / (1 - tax_rate) + interest_exp

# Simplified CFO (indirect)
cfo = ni + dep_amort + wc_investment

# Net fixed capital investment (positive = cash spent)
fc_inv_pos = (-capex) - proceeds_sale  # 600 - 50 = 550
wc_inv_pos = -wc_investment            # 90

print("=" * 55)
print("INPUT DATA")
print("=" * 55)
print(f"  Net Income              {ni:>10,.0f}")
print(f"  Depreciation & Amort    {dep_amort:>10,.0f}")
print(f"  Interest Expense        {interest_exp:>10,.0f}")
print(f"  Tax Rate                {tax_rate:>10.0%}")
print(f"  Capital Expenditures    {capex:>10,.0f}")
print(f"  Proceeds from Sales     {proceeds_sale:>10,.0f}")
print(f"  WC Investment           {wc_investment:>10,.0f}")
print(f"  Net Borrowing           {net_borrowing:>10,.0f}")
print(f"  EBIT (back-solved)      {ebit:>10,.0f}")
print(f"  CFO (simplified)        {cfo:>10,.0f}")

print(f"\n{'=' * 55}")
print("FCFF -- THREE APPROACHES")
print("=" * 55)

# Approach 1: From CFO
fcff_1 = cfo + interest_exp * (1 - tax_rate) - fc_inv_pos
print(f"\n  Approach 1: From CFO")
print(f"    CFO                          {cfo:>10,.0f}")
print(f"    + Int(1-t)                   {interest_exp*(1-tax_rate):>10,.0f}")
print(f"    - FCInv                      {-fc_inv_pos:>10,.0f}")
print(f"    FCFF                         {fcff_1:>10,.0f}")

# Approach 2: From EBIT
fcff_2 = ebit * (1 - tax_rate) + dep_amort - fc_inv_pos - wc_inv_pos
print(f"\n  Approach 2: From EBIT")
print(f"    EBIT(1-t)                    {ebit*(1-tax_rate):>10,.0f}")
print(f"    + Dep                        {dep_amort:>10,.0f}")
print(f"    - FCInv                      {-fc_inv_pos:>10,.0f}")
print(f"    - WCInv                      {-wc_inv_pos:>10,.0f}")
print(f"    FCFF                         {fcff_2:>10,.0f}")

# Approach 3: From NI
fcff_3 = ni + dep_amort + interest_exp * (1 - tax_rate) - fc_inv_pos - wc_inv_pos
print(f"\n  Approach 3: From NI")
print(f"    NI                           {ni:>10,.0f}")
print(f"    + NCC (Dep)                  {dep_amort:>10,.0f}")
print(f"    + Int(1-t)                   {interest_exp*(1-tax_rate):>10,.0f}")
print(f"    - FCInv                      {-fc_inv_pos:>10,.0f}")
print(f"    - WCInv                      {-wc_inv_pos:>10,.0f}")
print(f"    FCFF                         {fcff_3:>10,.0f}")

print(f"\n  All three equal? {np.allclose([fcff_1, fcff_2], fcff_3, atol=ATOL)}")

# FCFE
fcfe = fcff_3 - interest_exp * (1 - tax_rate) + net_borrowing
print(f"\n{'=' * 55}")
print("FCFE")
print("=" * 55)
print(f"  FCFF                           {fcff_3:>10,.0f}")
print(f"  - Int(1-t)                     {-interest_exp*(1-tax_rate):>10,.0f}")
print(f"  + Net Borrowing                {net_borrowing:>10,.0f}")
print(f"  FCFE                           {fcfe:>10,.0f}")

## 7. Cash Flow Quality Indicators

### Why Cash Flow Quality Matters

High-quality earnings are characterised by a strong and persistent relationship between reported net income and operating cash flow. When NI and CFO move together over time, it suggests that reported earnings are backed by real cash generation — the company is genuinely creating economic value, and its accounting faithfully reflects that reality.

When NI and CFO diverge — especially when NI grows while CFO stagnates or declines — it is a warning sign. The divergence indicates that an increasing proportion of earnings is composed of **accruals** (non-cash accounting entries) rather than actual cash. This can arise innocently (e.g., rapid growth requires working capital investment) or ominously (e.g., aggressive revenue recognition, channel stuffing, or capitalisation of operating expenses).

### Key Quality Metrics

#### 1. CFO-to-Net-Income Ratio

$$\text{CFO/NI Ratio} = \frac{\text{CFO}}{\text{Net Income}}$$

Interpretation guidelines:
- **Healthy range**: 1.0 to 1.5 — CFO should exceed NI because depreciation (a large non-cash charge) is added back in the indirect method. For capital-intensive businesses, the ratio may be 1.5 or higher.
- **Warning zone**: 0.5 to 1.0 — Working capital is consuming a significant portion of the cash that "should" be generated. Investigate the causes.
- **Red flag**: Below 0.5, or consistently declining — Earnings are increasingly disconnected from cash reality. The company may be using aggressive accounting.
- **Critical**: Negative CFO with positive NI — The company is reporting profits it cannot collect as cash. Potential fraud or impending liquidity crisis.

#### 2. Accruals Ratio (Balance Sheet Approach)

$$\text{Accruals Ratio}_{BS} = \frac{\text{NI} - \text{CFO}}{\text{Average Net Operating Assets}}$$

where Net Operating Assets = Operating Assets - Operating Liabilities (excludes cash and debt).

A high accruals ratio means a large portion of earnings is non-cash — i.e., earnings are "accrual-heavy." Research by Richard Sloan (1996, *The Accounting Review*) demonstrated that companies with high accruals ratios tend to experience **negative abnormal stock returns** in subsequent years as accruals reverse. This finding, known as the **accrual anomaly**, has been one of the most robust findings in empirical accounting research.

#### 3. Accruals Ratio (Cash Flow Approach)

$$\text{Accruals Ratio}_{CF} = \frac{\text{NI} - \text{CFO} - \text{CFI}}{\text{Average Net Operating Assets}}$$

This variant also removes investing cash flows, isolating the non-cash, non-investment component of earnings. It is sometimes preferred because it avoids penalising companies that are investing heavily for growth.

> **Key Concept:** The accruals ratio is one of the most powerful tools in fundamental analysis. High accruals today tend to mean lower earnings tomorrow. This is known as the **accrual anomaly** and has been documented extensively in academic finance literature. Many quantitative investment strategies include an accrual factor.

### Comprehensive Red Flags in Cash Flow Analysis

| Red Flag | What It Suggests | Historical Example |
|----------|-----------------|-------------------|
| NI > 0 but CFO < 0 for multiple periods | Earnings are not backed by cash; possible revenue manipulation | Enron (2000-2001) |
| CFO/NI ratio declining steadily over time | Earnings quality is deteriorating; accruals are growing | WorldCom (1999-2001) |
| Large and growing gap between NI and CFO | Aggressive working capital or revenue recognition | Lucent Technologies (late 1990s) |
| Capitalising operating expenses | Items that should reduce CFO are shifted to CFI, inflating both CFO and total assets | WorldCom (capitalised line costs) |
| Unusual items in "other" operating activities | Potential reclassification to inflate CFO | Various |
| Cash from customers declining while revenue grows | Receivables are being stuffed (revenue recognised on sales that may never be collected) | Sunbeam (1997-1998) |
| Dramatic increase in days sales outstanding (DSO) | Customers are not paying — or "sales" are fictitious | Tyco International |
| Operating cash flow boosted by securitisation or factoring | Company is selling receivables to convert NI to CFO artificially | Various financial institutions |

### The Accrual Spectrum

Companies can be placed on a spectrum from "cash-heavy" to "accrual-heavy" earnings:

$$\text{Cash-heavy} \longleftarrow \text{CFO} \gg \text{NI} \quad|\quad \text{CFO} \approx \text{NI} \quad|\quad \text{CFO} \ll \text{NI} \longrightarrow \text{Accrual-heavy}$$

- **Cash-heavy** (CFO >> NI): Typically mature, capital-intensive businesses with large depreciation. Earnings quality is usually high.
- **Balanced** (CFO $\approx$ NI): Healthy relationship. The company's accounting faithfully reflects its cash reality.
- **Accrual-heavy** (CFO << NI): The company is booking significant income that does not correspond to cash collection. May be growing rapidly (innocent) or manipulating earnings (concerning).

> **CFA Exam Tip:** The CFA exam often presents a multi-year dataset and asks you to identify which company has the highest earnings quality. The company with the most consistent CFO/NI ratio (closest to and above 1.0) typically has the highest quality. Watch for companies where the ratio is declining — that's usually the "worst quality" answer choice.

> **Common Mistake:** A CFO/NI ratio above 1.0 is not automatically good — it could indicate that the company is running down its working capital (collecting receivables faster than it replaces them, a sign of shrinking business). Or it could reflect very large depreciation on assets that will need replacement. Always look at the components, not just the ratio.

In [ ]:
# ── Generate 10-year data: a company with diverging NI and CFO ──────────

years = np.arange(2015, 2025)
n_years = len(years)

# Net income grows steadily (aggressive accounting)
ni_series = 200 + 30 * np.arange(n_years) + rng.normal(0, 10, n_years)

# CFO grows more slowly and eventually stagnates
cfo_series = 220 + 15 * np.arange(n_years) + rng.normal(0, 15, n_years)
cfo_series[7:] -= np.array([30, 60, 90])

# Compute quality metrics
cfo_ni_ratio = cfo_series / ni_series
accruals = ni_series - cfo_series

print("=" * 75)
print("10-YEAR CASH FLOW QUALITY ANALYSIS")
print("=" * 75)
print(f"{'Year':<6} {'NI':>8} {'CFO':>8} {'Accrual':>9} {'CFO/NI':>8} {'Signal':>12}")
print("-" * 75)
for i in range(n_years):
    signal = "OK" if cfo_ni_ratio[i] >= 0.9 else ("WATCH" if cfo_ni_ratio[i] >= 0.7 else "RED FLAG")
    print(f"{years[i]:<6} {ni_series[i]:>8,.0f} {cfo_series[i]:>8,.0f} "
          f"{accruals[i]:>+9,.0f} {cfo_ni_ratio[i]:>8.2f} {signal:>12}")

### Visualising Earnings Quality Deterioration

The left panel shows NI and CFO over time, with the shaded region highlighting the growing "accrual gap." The right panel is a scatter plot of CFO vs NI for each year, coloured by year. Points below the 45-degree line indicate that CFO is less than NI — i.e., earnings are accrual-heavy.

A healthy company would show points clustered along or above the 45-degree line. The pattern below — points drifting further below the line in recent years — is a classic warning sign of deteriorating earnings quality.

In [ ]:
# ── Scatter plot: CFO vs NI over 10 years ────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: time series
ax = axes[0]
ax.plot(years, ni_series, 'o-', color=SECONDARY, label='Net Income', linewidth=2, markersize=6)
ax.plot(years, cfo_series, 's-', color=PRIMARY, label='CFO', linewidth=2, markersize=6)
ax.fill_between(years, ni_series, cfo_series, alpha=0.15, color='red',
                where=ni_series > cfo_series, label='Accrual gap')
ax.set_xlabel('Year')
ax.set_ylabel('Amount ($)')
ax.set_title('NI vs CFO Over Time\n(Growing divergence = quality concern)')
ax.legend()

# Right: scatter
ax = axes[1]
scatter = ax.scatter(ni_series, cfo_series, c=years, cmap='coolwarm',
                     s=100, edgecolors='black', linewidths=1, zorder=5)
lims = [min(ni_series.min(), cfo_series.min()) - 20,
        max(ni_series.max(), cfo_series.max()) + 20]
ax.plot(lims, lims, '--', color='grey', alpha=0.7, label='CFO = NI line')
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_xlabel('Net Income ($)')
ax.set_ylabel('CFO ($)')
ax.set_title('CFO vs NI Scatter\n(Points below line = poor quality)')
ax.legend()
plt.colorbar(scatter, ax=ax, label='Year')

plt.tight_layout()
plt.show()

## 8. Company Lifecycle Stage Analysis

### The Eight Sign Patterns

The combination of positive (+) or negative (-) signs for CFO, CFI, and CFF creates $2^3 = 8$ possible patterns. Each pattern tells a story about where the company sits in its lifecycle and what strategic choices it is making:

| Pattern | CFO | CFI | CFF | Lifecycle Stage | Detailed Description |
|---------|-----|-----|-----|----------------|---------------------|
| 1 | + | - | + | **Early Growth** | Operations are cash-positive but insufficient to fund aggressive investment. The company supplements with external capital (debt or equity). Common for companies post-IPO scaling rapidly. |
| 2 | + | - | - | **Mature / Cash Cow** | The gold standard. Operations generate more than enough to fund investments, and excess cash is returned via debt repayment, buybacks, or dividends. Think Apple, Johnson & Johnson. |
| 3 | + | + | - | **Restructuring / Downsizing** | Operations are healthy, but the company is shrinking its asset base (selling divisions, closing plants). Proceeds plus CFO are used to deleverage or return capital. |
| 4 | + | + | + | **Unusual** | All three positive is rare. Could indicate a company building a war chest (selling assets AND raising capital despite positive CFO). May precede a major acquisition. |
| 5 | - | - | + | **Start-up / Young Growth** | Burning cash operationally (product not yet profitable). Investing heavily in the future. Entirely funded by external capital (VC, IPO proceeds, debt). Classic Silicon Valley pattern. |
| 6 | - | + | + | **Distressed / Liquidating** | Operations are cash-negative (the business is failing). Selling assets and raising emergency capital to stay afloat. A company in crisis. |
| 7 | - | - | - | **Severe Distress** | Negative across all three — the company is bleeding cash from operations, still investing (perhaps contractually obligated), and somehow repaying capital. Unsustainable. |
| 8 | - | + | - | **Late Decline** | Operations burn cash. The company sells assets to fund debt repayment. A controlled wind-down or involuntary liquidation. |

> **Key Concept:** The most common healthy pattern is **Pattern 2** (CFO+, CFI-, CFF-): the company generates enough operating cash to fund its investments and return capital to shareholders. This is the hallmark of a mature, well-managed company. If you see this pattern with growing CFO, you are likely looking at a high-quality business.

### Typical Lifecycle Progression

A company's cash flow signature typically evolves through these stages:

**Stage 1 — Start-up**: CFO(-), CFI(-), CFF(+)
The company is pre-profit. It burns cash developing products, building infrastructure, and hiring. All of this is funded by venture capital, angel investors, or IPO proceeds. Revenue is minimal or zero.

**Stage 2 — Growth**: CFO(+), CFI(-), CFF(+)
Operations turn cash-positive, but growth requires massive investment (new factories, geographic expansion, R&D). Operating cash flow alone is insufficient, so the company continues to raise external capital.

**Stage 3 — Maturity**: CFO(+), CFI(-), CFF(-)
The business is established and generates abundant cash. Investment needs moderate (maintenance capex plus selective growth). Excess cash is returned through dividends, buybacks, and debt reduction.

**Stage 4 — Decline**: CFO(-/+), CFI(+), CFF(-)
The core business weakens. The company sells assets (positive CFI) to meet obligations and reduce debt. If operations remain slightly positive, the decline may be managed; if negative, the company is in distress.

> **CFA Exam Tip:** The CFA exam may give you three companies with different CFO/CFI/CFF sign combinations and ask you to identify which is a start-up, mature company, or declining company. Use the pattern table above — it's tested directly.

### Caution on Over-Simplification

While these patterns provide useful heuristics, they should not be applied mechanically:

- A mature company with CFO(+), CFI(-), CFF(+) might simply be making a large strategic acquisition funded by a bond issuance — not reverting to "growth stage." Amazon's pattern fluctuated significantly during its AWS expansion years.

- A company with CFO(-) for a single year might have a timing issue (e.g., large one-time tax payment, legal settlement) rather than an operational problem.

- Capital-intensive industries (utilities, telecoms, real estate) will almost always have large negative CFI, which is normal for their business model and does not indicate "aggressive investment."

- Cyclical industries (mining, oil & gas) may show wildly different patterns depending on commodity prices, without any change in management quality.

> **Common Mistake:** Classifying a company's lifecycle stage from a single year of cash flow data. Always look at multi-year trends. One year's pattern can be distorted by one-time events, and the trajectory matters more than any single snapshot.

In [ ]:
# ── Classify sample companies by cash flow sign pattern ──────────────────

companies = {
    'TechStart Inc.':     {'CFO': -120, 'CFI': -300, 'CFF':  450},
    'GrowthCo Ltd.':      {'CFO':  180, 'CFI': -400, 'CFF':  250},
    'SteadyMfg Corp.':    {'CFO':  500, 'CFI': -200, 'CFF': -250},
    'OldRetail Inc.':     {'CFO':  -50, 'CFI':  150, 'CFF': -120},
    'CrisisBank Ltd.':    {'CFO': -200, 'CFI':  100, 'CFF':  150},
    'RestructureCo.':     {'CFO':  300, 'CFI':  200, 'CFF': -450},
    'DiversiGlobal':      {'CFO':  400, 'CFI':  100, 'CFF':  200},
    'TerminalCo.':        {'CFO':  -80, 'CFI': -100, 'CFF':  -30},
}

pattern_map = {
    ('+', '-', '+'): 'Early Growth',
    ('+', '-', '-'): 'Mature / Cash Cow',
    ('+', '+', '-'): 'Restructuring',
    ('+', '+', '+'): 'Unusual',
    ('-', '-', '+'): 'Start-up',
    ('-', '+', '+'): 'Distressed',
    ('-', '-', '-'): 'Severe Distress',
    ('-', '+', '-'): 'Late Decline',
}

print("=" * 85)
print("COMPANY LIFECYCLE CLASSIFICATION BY CASH FLOW PATTERN")
print("=" * 85)
print(f"{'Company':<20} {'CFO':>6} {'CFI':>6} {'CFF':>6} {'Pattern':>10} {'Stage':<20}")
print("-" * 85)
for name, cfs in companies.items():
    signs = tuple('+' if cfs[k] >= 0 else '-' for k in ['CFO', 'CFI', 'CFF'])
    stage = pattern_map.get(signs, 'Unknown')
    pattern_str = '/'.join(signs)
    print(f"{name:<20} {cfs['CFO']:>6,} {cfs['CFI']:>6,} {cfs['CFF']:>6,} "
          f"{pattern_str:>10} {stage:<20}")

### Heatmap Visualisation

The heatmap below encodes positive cash flows in green and negative in red for each company across the three activities. The lifecycle stage labels are shown on the right. This type of visualisation makes it easy to spot patterns at a glance — for example, the "Mature / Cash Cow" pattern (green-red-red) immediately stands out from the "Start-up" pattern (red-red-green).

In [ ]:
# ── Heatmap of cash flow patterns ────────────────────────────────────────

comp_names = list(companies.keys())
activities = ['CFO', 'CFI', 'CFF']
n_comp = len(comp_names)

sign_matrix = np.zeros((n_comp, 3))
for i, name in enumerate(comp_names):
    for j, act in enumerate(activities):
        sign_matrix[i, j] = 1 if companies[name][act] >= 0 else -1

fig, ax = plt.subplots(figsize=(8, 7))
cmap = plt.cm.RdYlGn
im = ax.imshow(sign_matrix, cmap=cmap, aspect='auto', vmin=-1.5, vmax=1.5)

ax.set_xticks(range(3))
ax.set_xticklabels(activities, fontsize=13, fontweight='bold')
ax.set_yticks(range(n_comp))
ax.set_yticklabels(comp_names, fontsize=11)

for i in range(n_comp):
    for j in range(3):
        val = companies[comp_names[i]][activities[j]]
        sign = '+' if val >= 0 else '-'
        color = 'white' if abs(sign_matrix[i, j]) > 0.5 else 'black'
        ax.text(j, i, f'{sign}\n${abs(val):,}', ha='center', va='center',
                fontsize=10, fontweight='bold', color=color)

for i, name in enumerate(comp_names):
    signs = tuple('+' if companies[name][k] >= 0 else '-' for k in activities)
    stage = pattern_map.get(signs, '?')
    ax.text(3.1, i, stage, ha='left', va='center', fontsize=9,
            fontstyle='italic', color='black')

ax.set_title('Cash Flow Sign Patterns and Lifecycle Stages', fontsize=14, pad=15)
ax.set_xlim(-0.5, 2.5)
plt.tight_layout()
plt.show()

## 9. Comprehensive Worked Example

### Problem Setup

We are given two years of balance sheets and a Year 2 income statement for **Apex Manufacturing Corp.** Our task:

1. Construct the **complete indirect-method CFO** from scratch
2. Build a **waterfall chart** showing the reconciliation from Net Income to CFO
3. Compute **FCFF** and **FCFE**
4. Verify the cash flow identity: $\Delta$Cash = CFO + CFI + CFF

This is the type of comprehensive problem that appears frequently on the CFA Level 1 exam and in financial analyst interviews. It requires integrating knowledge from all previous sections.

> **Key Concept:** In practice, constructing the cash flow statement requires a systematic approach: (1) start with NI, (2) add back all non-cash charges, (3) remove non-operating gains/losses, (4) compute every working capital change from balance sheet deltas, (5) verify the total against the actual change in cash.

### Given Data

**Income Statement — Apex Manufacturing Corp., Year 2:**

| Item | Amount ($) |
|------|-----------|
| Revenue | 12,500 |
| Cost of Goods Sold | (7,500) |
| **Gross Profit** | **5,000** |
| Depreciation & Amortisation | (800) |
| SG&A Expenses | (1,500) |
| Gain on Sale of Equipment | 120 |
| Interest Expense | (300) |
| **Pre-tax Income** | **2,520** |
| Tax Expense (25%) | (630) |
| **Net Income** | **1,890** |

**Comparative Balance Sheets:**

| Item | Year 1 ($) | Year 2 ($) |
|------|-----------|-----------|
| **Assets** | | |
| Cash | 500 | (to be determined) |
| Accounts Receivable | 1,800 | 2,100 |
| Inventory | 2,500 | 2,350 |
| Prepaid Expenses | 200 | 260 |
| PP&E (net) | 8,000 | 8,600 |
| **Liabilities** | | |
| Accounts Payable | 1,400 | 1,550 |
| Accrued Liabilities | 600 | 720 |
| Taxes Payable | 300 | 250 |
| Short-term Debt | 500 | 500 |
| Long-term Debt | 3,000 | 3,400 |
| **Equity** | | |
| Common Stock | 4,000 | 4,000 |
| Retained Earnings | 3,200 | (to be determined) |

**Additional information:**
- Capital expenditures during Year 2: $1,520
- Proceeds from sale of equipment: $220 (book value of sold equipment was $100, hence gain of $120)
- Dividends paid: $400
- Net new long-term debt issued: $400

**PP&E Rollforward Verification:**

$$\text{End PP\&E} = \text{Beg PP\&E} + \text{Capex} - \text{Depreciation} - \text{BV of Assets Sold}$$
$$8,600 = 8,000 + 1,520 - 800 - 120 \quad \checkmark$$

Wait — book value of sold equipment is $100, not $120 (the $120 is the gain). Let us verify:
$$8,000 + 1,520 - 800 - 100 = 8,620 \neq 8,600$$

This means the book value of sold equipment is actually $120 (cost minus accumulated depreciation at time of sale = $120). The gain = proceeds - book value = $220 - $120 = $100. We will use gain = $100 in our calculations.

Corrected: Gain on sale = $100, and PP&E rollforward: $8,000 + 1,520 - 800 - 120 = 8,600. $\checkmark$

> **CFA Exam Tip:** On the exam, you may need to back-solve for capex or book value of disposed assets using the PP&E rollforward equation. Always verify that your numbers are internally consistent before proceeding to the cash flow statement.

In [ ]:
# ── Comprehensive worked example: Apex Manufacturing Corp. ──────────────

# Income statement (corrected gain = 100)
apex_revenue = 12500.0
apex_cogs = -7500.0
apex_dep = -800.0
apex_sga = -1500.0
apex_gain_sale = 100.0  # corrected: proceeds 220 - BV 120 = 100
apex_interest = -300.0
apex_tax_rate = 0.25

apex_pretax = (apex_revenue + apex_cogs + apex_dep + apex_sga
               + apex_gain_sale + apex_interest)
apex_tax = -apex_pretax * apex_tax_rate
apex_ni = apex_pretax + apex_tax

print("=" * 55)
print("APEX MANUFACTURING -- INCOME STATEMENT (Year 2)")
print("=" * 55)
print(f"  Revenue                     {apex_revenue:>10,.0f}")
print(f"  COGS                        {apex_cogs:>10,.0f}")
print(f"  Gross Profit                {apex_revenue + apex_cogs:>10,.0f}")
print(f"  Depreciation & Amort        {apex_dep:>10,.0f}")
print(f"  SG&A                        {apex_sga:>10,.0f}")
print(f"  Gain on Sale                {apex_gain_sale:>10,.0f}")
print(f"  Interest Expense            {apex_interest:>10,.0f}")
print(f"  Pre-tax Income              {apex_pretax:>10,.0f}")
print(f"  Tax Expense                 {apex_tax:>10,.0f}")
print(f"  {'─' * 35}")
print(f"  Net Income                  {apex_ni:>10,.0f}")

In [ ]:
# ── Balance sheets and working capital changes ──────────────────────────

bs = {
    'Accounts Receivable':  (1800, 2100),
    'Inventory':            (2500, 2350),
    'Prepaid Expenses':     (200,  260),
    'Accounts Payable':     (1400, 1550),
    'Accrued Liabilities':  (600,  720),
    'Taxes Payable':        (300,  250),
}

is_current_asset = {
    'Accounts Receivable': True,
    'Inventory': True,
    'Prepaid Expenses': True,
    'Accounts Payable': False,
    'Accrued Liabilities': False,
    'Taxes Payable': False,
}

print("=" * 70)
print("BALANCE SHEET CHANGES AND WORKING CAPITAL ADJUSTMENTS")
print("=" * 70)
print(f"{'Item':<25} {'Year 1':>8} {'Year 2':>8} {'Change':>8} {'Cash Adj':>10}")
print("-" * 70)

total_wc_adj = 0.0
wc_detail = {}
for item, (y1, y2) in bs.items():
    change = y2 - y1
    cash_adj = -change if is_current_asset[item] else change
    total_wc_adj += cash_adj
    wc_detail[item] = cash_adj
    print(f"{item:<25} {y1:>8,.0f} {y2:>8,.0f} {change:>+8,.0f} {cash_adj:>+10,.0f}")

print("-" * 70)
print(f"{'Total WC Adjustment':<25} {'':>8} {'':>8} {'':>8} {total_wc_adj:>+10,.0f}")

### Building the Complete Indirect-Method CFO

Now we assemble all the pieces: start with net income, add non-cash adjustments, remove non-operating items, and apply working capital changes.

In [ ]:
# ── Build complete indirect-method CFO ───────────────────────────────────

print("=" * 55)
print("CFO -- INDIRECT METHOD (Complete)")
print("=" * 55)
print(f"  Net Income                    {apex_ni:>10,.0f}")

dep_addback = -apex_dep
print(f"\n  Non-cash adjustments:")
print(f"    + Depreciation & Amort      {dep_addback:>10,.0f}")
print(f"    - Gain on sale of equip     {-apex_gain_sale:>10,.0f}")

print(f"\n  Working capital changes:")
for item, adj in wc_detail.items():
    print(f"    {item:<27} {adj:>+10,.0f}")

apex_cfo = apex_ni + dep_addback + (-apex_gain_sale) + total_wc_adj
print(f"\n  {'─' * 35}")
print(f"  CFO                           {apex_cfo:>10,.0f}")

# CFI and CFF
apex_capex = -1520.0
apex_proceeds = 220.0
apex_dividends = -400.0
apex_net_borrowing = 400.0

apex_cfi = apex_capex + apex_proceeds
apex_cff = apex_net_borrowing + apex_dividends

print(f"\n{'=' * 55}")
print("CFI")
print("=" * 55)
print(f"  Capital expenditures          {apex_capex:>10,.0f}")
print(f"  Proceeds from sale            {apex_proceeds:>10,.0f}")
print(f"  CFI                           {apex_cfi:>10,.0f}")

print(f"\n{'=' * 55}")
print("CFF")
print("=" * 55)
print(f"  Net borrowing                 {apex_net_borrowing:>10,.0f}")
print(f"  Dividends paid                {apex_dividends:>10,.0f}")
print(f"  CFF                           {apex_cff:>10,.0f}")

delta_cash = apex_cfo + apex_cfi + apex_cff
print(f"\n{'=' * 55}")
print("CASH RECONCILIATION")
print("=" * 55)
print(f"  CFO + CFI + CFF = Delta Cash  {delta_cash:>10,.0f}")
print(f"  Beginning Cash                {500:>10,.0f}")
print(f"  Ending Cash                   {500 + delta_cash:>10,.0f}")

### FCFF and FCFE for Apex Manufacturing

Using the formulas from Section 6, we compute both free cash flow measures.

In [ ]:
# ── FCFF and FCFE for Apex ───────────────────────────────────────────────

fc_inv_net = (-apex_capex) - apex_proceeds  # 1520 - 220 = 1300
apex_fcff = apex_cfo + abs(apex_interest) * (1 - apex_tax_rate) - fc_inv_net
apex_fcfe = apex_fcff - abs(apex_interest) * (1 - apex_tax_rate) + apex_net_borrowing

print("=" * 55)
print("FREE CASH FLOW -- APEX MANUFACTURING")
print("=" * 55)
print(f"  CFO                           {apex_cfo:>10,.0f}")
print(f"  + Int(1-t)                    {abs(apex_interest)*(1-apex_tax_rate):>10,.0f}")
print(f"  - FCInv (net capex)           {-fc_inv_net:>10,.0f}")
print(f"  FCFF                          {apex_fcff:>10,.0f}")
print(f"\n  FCFF                          {apex_fcff:>10,.0f}")
print(f"  - Int(1-t)                    {-abs(apex_interest)*(1-apex_tax_rate):>10,.0f}")
print(f"  + Net Borrowing               {apex_net_borrowing:>10,.0f}")
print(f"  FCFE                          {apex_fcfe:>10,.0f}")

### Waterfall Chart: NI to CFO Reconciliation

The waterfall chart below visualises each step in the indirect method, showing how net income is transformed into operating cash flow through non-cash adjustments and working capital changes. Blue bars represent the starting (NI) and ending (CFO) totals. Green bars are positive adjustments (increasing CFO), and coral bars are negative adjustments (decreasing CFO).

This type of chart is commonly used in investor presentations and analyst reports to communicate the "bridge" from earnings to cash flow.

In [ ]:
# ── Waterfall chart: NI to CFO reconciliation ───────────────────────────

labels = ['Net\nIncome', '+ Deprec', '- Gain on\nSale']
values = [apex_ni, dep_addback, -apex_gain_sale]

short_names = {
    'Accounts Receivable': 'A/R', 'Inventory': 'Inv',
    'Prepaid Expenses': 'Prepaid', 'Accounts Payable': 'A/P',
    'Accrued Liabilities': 'Accrued', 'Taxes Payable': 'Tax Pay'
}
for item, adj in wc_detail.items():
    labels.append(short_names[item])
    values.append(adj)

labels.append('CFO')

# Compute waterfall positions
n = len(labels)
cumulative = np.zeros(n)
cumulative[0] = values[0]
for i in range(1, n - 1):
    cumulative[i] = cumulative[i-1] + values[i]

# Final bar = total CFO
cfo_total = cumulative[n - 2]

# Bar bottoms and heights
bottoms = np.zeros(n)
heights = np.zeros(n)

# First bar: starts at 0, height = NI
bottoms[0] = 0
heights[0] = values[0]

# Middle bars: waterfall steps
for i in range(1, n - 1):
    if values[i] >= 0:
        bottoms[i] = cumulative[i-1]
        heights[i] = values[i]
    else:
        bottoms[i] = cumulative[i]
        heights[i] = -values[i]

# Last bar: starts at 0, height = total CFO
bottoms[-1] = 0
heights[-1] = cfo_total

# Colours
bar_colors = [PRIMARY]  # NI
for i in range(1, n - 1):
    bar_colors.append(TERTIARY if values[i] >= 0 else SECONDARY)
bar_colors.append(PRIMARY)  # CFO

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.bar(range(n), heights, bottom=bottoms, color=bar_colors,
              edgecolor='white', linewidth=1.5, width=0.7)

# Connect bars with dashed lines
for i in range(n - 2):
    ax.plot([i + 0.35, i + 0.65], [cumulative[i], cumulative[i]],
            color='black', linewidth=0.8, linestyle='--')

# Labels
for i, bar in enumerate(bars):
    if i == 0 or i == n - 1:
        display = values[0] if i == 0 else cfo_total
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_y() + bar.get_height()/2,
                f'${display:,.0f}', ha='center', va='center',
                fontweight='bold', fontsize=9, color='white')
    else:
        y_pos = bar.get_y() + bar.get_height()/2
        ax.text(bar.get_x() + bar.get_width()/2, y_pos,
                f'${values[i]:+,.0f}', ha='center', va='center',
                fontweight='bold', fontsize=9,
                color='white' if bar.get_height() > 80 else 'black')

ax.set_xticks(range(n))
ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Amount ($)')
ax.set_title('Waterfall: Net Income to CFO Reconciliation (Apex Manufacturing)',
             fontsize=13)
ax.axhline(0, color='black', linewidth=0.8)

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=PRIMARY, label='NI / CFO Total'),
                   Patch(facecolor=TERTIARY, label='Positive adjustment'),
                   Patch(facecolor=SECONDARY, label='Negative adjustment')]
ax.legend(handles=legend_elements, loc='upper right')

plt.tight_layout()
plt.show()

print(f"Net Income: ${apex_ni:,.0f}  -->  CFO: ${cfo_total:,.0f}")
print(f"Total adjustments: ${cfo_total - apex_ni:+,.0f}")

## 10. References

1. **CFA Institute** (2024). *CFA Program Curriculum Level I: Financial Statement Analysis*. CFA Institute.

2. **International Accounting Standards Board** (2023). *IAS 7 — Statement of Cash Flows*. IFRS Foundation.

3. **Financial Accounting Standards Board** (2023). *ASC 230 — Statement of Cash Flows*. FASB.

4. **Sloan, R.G.** (1996). "Do Stock Prices Fully Reflect Information in Accruals and Cash Flows about Future Earnings?" *The Accounting Review*, 71(3), 289-315.

5. **Mulford, C.W. & Comiskey, E.E.** (2005). *Creative Cash Flow Reporting: Uncovering Sustainable Financial Performance*. John Wiley & Sons.

6. **Fridson, M.S. & Alvarez, F.** (2011). *Financial Statement Analysis: A Practitioner's Guide*. 4th Edition. Wiley.

7. **Robinson, T.R., Henry, E., Pirie, W.L., & Broihahn, M.A.** (2020). *International Financial Statement Analysis*. 4th Edition. CFA Institute / Wiley.

8. **Penman, S.H.** (2013). *Financial Statement Analysis and Security Valuation*. 5th Edition. McGraw-Hill.

---

*Notebook created for educational purposes. All financial data is synthetic.*